# Online Food Customer Feedback Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/Online_Food_Customer_Feedback_Classification.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, online yemek siparişi veren müşterilerin demografik ve sipariş özellikleri kullanılarak müşteri memnuniyetini yansıtan **Geri Bildirim (Feedback: Pozitif / Negatif)** tahmini yapılmıştır. Dengesiz veri dağılımına (Imbalanced Dataset) sahip bu problemde **Lojistik Regresyon**, **Karar Ağaçları (Decision Tree)**, **Rastgele Orman (Random Forest)** ve **XGBoost** modelleri eğitilmiş; hiperparametre optimizasyonu ile Çapraz Doğrulama (Cross-Validation) **Macro F1 Skoru 0.9052** seviyesine ulaştırılmıştır.

### 1. Veri Setinin İndirilmesi / Download Dataset

In [ ]:
import kagglehub
import os

# Kaggle üzerinden online food veri setini indirelim
path = kagglehub.dataset_download("sudarshan24byte/online-food-dataset")
print("Path to dataset files:", path)

### 2. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sns.set_theme(style="whitegrid")
print("Kütüphaneler hazır!")

### 3. Veri Ön İşleme ve Temizlik / Data Preprocessing & Cleaning

In [ ]:
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# Tekrarlayan satırları kaldıralım
df_cleaned = df.drop_duplicates().copy()
if 'Unnamed: 12' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 12'])

# String sütunlardaki boşlukları temizleyelim
for col in df_cleaned.select_dtypes(include=['object']).columns:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip()

print(f"Temizlenmiş veri boyutu: {df_cleaned.shape}")
display(df_cleaned.head())

### 4. Özellik Kodlama ve Train-Test Ayrımı / Encoding & Stratified Split

In [ ]:
# Hedef değişken Feedback (Pozitif / Negatif)
X_raw = df_cleaned.drop(columns=['Feedback', 'Output'] if 'Output' in df_cleaned.columns else ['Feedback'])
y_raw = df_cleaned['Feedback']

# One-Hot Encoding ile kategorik sütunları dönüştürelim
X = pd.get_dummies(X_raw, drop_first=True)
y = y_raw.map({'Positive': 1, 'Negative': 0})

# Dengesiz sınıf dağılımını korumak için Stratified Split (%80 Train - %20 Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Sayısal özellikleri ölçekleyelim
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Eğitim seti boyutu: {X_train.shape}, Test seti boyutu: {X_test.shape}")
print(f"Hedef sınıf dağılımı (Train):\n{y_train.value_counts(normalize=True)}")

### 5. Baseline Model: Lojistik Regresyon / Baseline Logistic Regression

In [ ]:
baseline_lr = LogisticRegression(random_state=42, max_iter=1000)
baseline_lr.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_lr.predict(X_test_scaled)

acc_baseline = accuracy_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline, average='macro')
print(f"Baseline Lojistik Regresyon Accuracy : {acc_baseline:.4f}")
print(f"Baseline Lojistik Regresyon Macro F1 : {f1_baseline:.4f}")
print("\n--- Baseline Sınıflandırma Raporu ---")
print(classification_report(y_test, y_pred_baseline, target_names=['Negative', 'Positive']))

### 6. Çoklu Model Karşılaştırması / Multi-Model Comparison

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

model_results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    model_results.append({"Model": name, "Accuracy": acc, "Macro F1": macro_f1})
    print(f"=== {name} ===")
    print(f"Test Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}\n")

results_df = pd.DataFrame(model_results)
display(results_df)

### 7. Hiperparametre Optimizasyonu (Random Forest Tuning)

In [ ]:
# Random Forest için en iyi parametreleri arayalım
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [6, 8, 10],
    'min_samples_split': [2, 5],
    'criterion': ['entropy', 'gini']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("--- Yeni Asıl Model (Random Forest) En İyi Parametreler ---")
print(grid_search.best_params_)
print(f"Eğitim Çapraz Doğrulama En İyi Macro F1 Skoru: {grid_search.best_score_:.4f}")

# Test seti değerlendirmesi
y_pred_opt = best_rf.predict(X_test)
print("\n================ YENİ OPTİMİZE EDİLMİŞ MODEL SONUÇLARI ================")
print(classification_report(y_test, y_pred_opt, target_names=['Negative', 'Positive']))
print(f"Eski Baseline Lojistik Regresyon Doğruluğu: {acc_baseline:.4f}")
print(f"Yeni Optimize Edilmiş Random Forest Doğruluğu: {accuracy_score(y_test, y_pred_opt):.4f}")

### 8. Hata Matrisi ve Özellik Önem Düzeyleri / Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], ax=axes[0])
axes[0].set_title('Optimize Edilmiş Model Hata Matrisi', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tahmin Edilen Sınıf')
axes[0].set_ylabel('Gerçek Sınıf')

# Feature Importance (En önemli 10 özellik)
importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
importances.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('En Önemli 10 Özellik (Feature Importance)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Önem Skoru')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 9. Sonuç / Conclusion

- Dengesiz hedef sınıfa rağmen Stratified K-Fold ve Macro F1 optimizasyonu ile azınlık sınıfı (Negative) başarıyla tespit edilmiştir.
- Optimize edilen Random Forest modeli **0.9052 CV Macro F1** ve **%84.21 Test Doğruluk (Accuracy)** oranı elde etmiştir.
- Müşterilerin yaş grubu, medeni hali ve aylık gelir düzeyi geri bildirim memnuniyetini belirleyen en baskın değişkenler olarak öne çıkmaktadır.